# Lab 5, Day 2 — Pipeline, Features, and Model Selection

Build a leak-free `Pipeline`, get a cross-validated baseline, engineer features with a
stated hypothesis, compare models honestly, tune once, and evaluate on the test set
exactly once. See `Lab5_Day2_Instructions.md` for the full walkthrough.

This continues directly from Day 1's folder and split - not a restart.

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler

from pipeline import engineer, build_preprocessor, build_pipeline, get_feature_columns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.metrics import f1_score, classification_report, roc_auc_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier


## Before you start: watch leakage happen

Fit a `StandardScaler` on your *full* dataset (`X`, before any split) and print
`scaler.mean_`. Then fit a fresh one on `X_train` alone and print its `.mean_`. The
numbers differ. Before reading further, think about what that difference actually
means - which numbers were influenced by data your model should never have seen at
fit time? That's the entire argument for wrapping every fitted step in a `Pipeline`,
which is what you're about to build.

In [2]:
# TODO: the leakage demo described above (optional to keep in your final notebook,
# but do it before writing any pipeline code)

In [ ]:
# TODO: reload yesterday's split with joblib.load("split.joblib"), or re-run Day 1's
# Step 5-6 if you didn't save it

# Reload yesterday's split
bundle = joblib.load("split.joblib")
X_train = bundle["X_train"]
X_test = bundle["X_test"]
y_train = bundle["y_train"]
y_test = bundle["y_test"]
print("Loaded split:", X_train.shape, X_test.shape)

# Leakage demo: scale means differ when test rows influence the fit
num_demo = ["Age", "Fare", "SibSp", "Parch"]
X_full = pd.concat([X_train, X_test], axis=0)
# Use only columns present and numeric; Age/Fare have NaNs fill temporarily for demo
X_full_num = X_full[num_demo].astype(float)
X_train_num = X_train[num_demo].astype(float)

sc_full = StandardScaler()
sc_full.fit(X_full_num.fillna(X_full_num.median()))
sc_train = StandardScaler()
sc_train.fit(X_train_num.fillna(X_train_num.median()))

print("scaler means (fit on FULL data):", np.round(sc_full.mean_, 4))
print("scaler means (fit on TRAIN only):", np.round(sc_train.mean_, 4))
print("difference:", np.round(sc_full.mean_ - sc_train.mean_, 4))
print("-> The full-data means were influenced by test rows. That is leakage.")


Loaded split: (1047, 9) (262, 9)
scaler means (fit on FULL data): [29.5032 33.2811  0.4989  0.385 ]
scaler means (fit on TRAIN only): [29.6032 34.0369  0.4852  0.4002]
difference: [-0.1    -0.7558  0.0137 -0.0152]
-> The full-data means were influenced by test rows. That is leakage.


## Step 1: The ColumnTransformer (`pipeline.py`)

In [4]:
# TODO: build_preprocessor(num_cols, cat_cols) - a numeric sub-pipeline (impute then
# scale) and a categorical sub-pipeline (impute then one-hot encode). Remember
# handle_unknown on the encoder - a category seen only at predict time must not crash.


# Engineer features on train and test separately (no fit — pure transforms)
X_train_fe = engineer(X_train)
X_test_fe = engineer(X_test)

# Drop free-text columns that engineer() already consumed
drop_after = [c for c in ["Name", "Cabin"] if c in X_train_fe.columns]
X_train_fe = X_train_fe.drop(columns=drop_after)
X_test_fe = X_test_fe.drop(columns=drop_after)

num_cols, cat_cols = get_feature_columns(X_train_fe)
print("numeric:", num_cols)
print("categorical:", cat_cols)
assert "Pclass" in cat_cols, "Pclass must be treated as categorical"
print("X_train_fe shape:", X_train_fe.shape)

numeric: ['Age', 'SibSp', 'Parch', 'Fare', 'family_size']
categorical: ['Pclass', 'Sex', 'Embarked', 'is_alone', 'title', 'has_cabin']
X_train_fe shape: (1047, 11)


In [5]:
# TODO: build_pipeline(num_cols, cat_cols, model) - pre + model, ready to fit

pre = build_preprocessor(num_cols, cat_cols)
pipe_lr = build_pipeline(num_cols, cat_cols, LogisticRegression(max_iter=1000, random_state=0))
print(pipe_lr)

Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'SibSp', 'Parch',
                                                   'Fare', 'family_size']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ign

## Step 2: Cross-validated baseline

In [6]:
# TODO: cross_val_score on the training set only, an appropriate metric for this
# target's class balance, and report BOTH the mean and the standard deviation

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

def cv_report(pipe, X, y, name="model"):
    f1 = cross_val_score(pipe, X, y, cv=cv, scoring="f1")
    auc = cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc")
    print(f"{name}: F1  mean={f1.mean():.4f}  std={f1.std():.4f}")
    print(f"{name}: AUC mean={auc.mean():.4f}  std={auc.std():.4f}")
    return f1, auc

print("=== Baseline: Logistic Regression (with engineered features) ===")
f1_base, auc_base = cv_report(pipe_lr, X_train_fe, y_train, "LogReg")

=== Baseline: Logistic Regression (with engineered features) ===
LogReg: F1  mean=0.7541  std=0.0290
LogReg: AUC mean=0.8528  std=0.0116


## Step 3: Engineer features, with a stated hypothesis first

In [7]:
# TODO: engineer(df) in pipeline.py - for each feature, write the hypothesis as a
# comment before the code. Then actually test whether it helped the CV score, and
# report the result either way, even if it didn't help.

def make_xy(frame, drop=None):
    f = frame.copy()
    if drop:
        f = f.drop(columns=[c for c in drop if c in f.columns])
    n, c = get_feature_columns(f)
    return f, n, c

configs = {
    "all engineered": [],
    "no family_size/is_alone": ["family_size", "is_alone"],
    "no title": ["title"],
    "no has_cabin": ["has_cabin"],
    "no engineered (raw only)": ["family_size", "is_alone", "title", "has_cabin"],
}

print("Ablation study (LogisticRegression, F1 under 5-fold CV):\n")
for name, drops in configs.items():
    Xa, na, ca = make_xy(X_train_fe, drops)
    pipe = build_pipeline(na, ca, LogisticRegression(max_iter=1000, random_state=0))
    f1 = cross_val_score(pipe, Xa, y_train, cv=cv, scoring="f1")
    print(f"  {name:30s}  F1={f1.mean():.4f} ± {f1.std():.4f}  (cols={list(Xa.columns)})")

Ablation study (LogisticRegression, F1 under 5-fold CV):

  all engineered                  F1=0.7541 ± 0.0290  (cols=['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'family_size', 'is_alone', 'title', 'has_cabin'])
  no family_size/is_alone         F1=0.7448 ± 0.0291  (cols=['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'title', 'has_cabin'])
  no title                        F1=0.7130 ± 0.0258  (cols=['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'family_size', 'is_alone', 'has_cabin'])
  no has_cabin                    F1=0.7488 ± 0.0327  (cols=['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'family_size', 'is_alone', 'title'])
  no engineered (raw only)        F1=0.7072 ± 0.0331  (cols=['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked'])


## Step 4: Compare at least three models

In [8]:
# TODO: cross-validate at least three different model types with the same
# preprocessing, and report mean + std for each. Are the differences bigger than the
# fold-to-fold noise?

models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=0),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=0),
    "GradientBoosting": GradientBoostingClassifier(random_state=0),
    "KNN": KNeighborsClassifier(n_neighbors=15),
}

results = {}
print("Model comparison (same preprocessing, 5-fold stratified CV):\n")
for name, model in models.items():
    pipe = build_pipeline(num_cols, cat_cols, model)
    f1 = cross_val_score(pipe, X_train_fe, y_train, cv=cv, scoring="f1")
    auc = cross_val_score(pipe, X_train_fe, y_train, cv=cv, scoring="roc_auc")
    results[name] = (f1, auc)
    print(f"  {name:20s}  F1={f1.mean():.4f} ± {f1.std():.4f}   AUC={auc.mean():.4f} ± {auc.std():.4f}")

# Pick best by mean F1 for tuning
best_name = max(results, key=lambda k: results[k][0].mean())
print(f"\nBest by mean F1: {best_name}")
print("Note: if gaps are smaller than fold std, models are statistically similar.")

Model comparison (same preprocessing, 5-fold stratified CV):

  LogisticRegression    F1=0.7541 ± 0.0290   AUC=0.8528 ± 0.0116
  RandomForest          F1=0.7302 ± 0.0311   AUC=0.8460 ± 0.0243
  GradientBoosting      F1=0.7462 ± 0.0519   AUC=0.8656 ± 0.0180
  KNN                   F1=0.7194 ± 0.0386   AUC=0.8503 ± 0.0208

Best by mean F1: LogisticRegression
Note: if gaps are smaller than fold std, models are statistically similar.


## Step 5: Tune the best model, then evaluate the test set exactly once

In [9]:
# TODO: GridSearchCV on training data only (remember the model__param prefix for
# a parameter inside a named pipeline step)

# GridSearchCV on training data only. Parameter names use the model__ prefix.
if best_name == "LogisticRegression":
    param_grid = {
        "model__C": [0.1, 0.5, 1.0, 2.0, 5.0],
        "model__solver": ["lbfgs"],
    }
    base_model = LogisticRegression(max_iter=2000, random_state=0)
elif best_name == "RandomForest":
    param_grid = {
        "model__n_estimators": [100, 200, 400],
        "model__max_depth": [3, 5, 8, None],
        "model__min_samples_leaf": [1, 2, 4],
    }
    base_model = RandomForestClassifier(random_state=0)
elif best_name == "GradientBoosting":
    param_grid = {
        "model__n_estimators": [50, 100, 200],
        "model__learning_rate": [0.05, 0.1, 0.2],
        "model__max_depth": [2, 3, 4],
    }
    base_model = GradientBoostingClassifier(random_state=0)
else:
    param_grid = {
        "model__n_neighbors": [5, 11, 15, 21],
        "model__weights": ["uniform", "distance"],
    }
    base_model = KNeighborsClassifier()

search_pipe = build_pipeline(num_cols, cat_cols, base_model)
grid = GridSearchCV(
    search_pipe,
    param_grid,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    refit=True,
)
grid.fit(X_train_fe, y_train)
print("Best params:", grid.best_params_)
print("Best CV F1:", round(grid.best_score_, 4))

Best params: {'model__C': 5.0, 'model__solver': 'lbfgs'}
Best CV F1: 0.7562


In [10]:
# TODO: the test set, touched here for the first and only time - report an
# appropriate metric. If the tuned model does no better than the baseline, that is a
# result to report, not a bug to hide.

best_pipe = grid.best_estimator_
y_pred = best_pipe.predict(X_test_fe)
y_proba = best_pipe.predict_proba(X_test_fe)[:, 1]

test_f1 = f1_score(y_test, y_pred)
test_auc = roc_auc_score(y_test, y_proba)
print(f"Test F1:  {test_f1:.4f}")
print(f"Test AUC: {test_auc:.4f}")
print()
print(classification_report(y_test, y_pred, digits=3))

baseline_pipe = build_pipeline(num_cols, cat_cols, LogisticRegression(max_iter=1000, random_state=0))
baseline_pipe.fit(X_train_fe, y_train)
base_pred = baseline_pipe.predict(X_test_fe)
print(f"Untuned LogReg test F1 (for reference): {f1_score(y_test, base_pred):.4f}")

Test F1:  0.7579
Test AUC: 0.8647

              precision    recall  f1-score   support

           0      0.837     0.889     0.862       162
           1      0.800     0.720     0.758       100

    accuracy                          0.824       262
   macro avg      0.819     0.804     0.810       262
weighted avg      0.823     0.824     0.822       262

Untuned LogReg test F1 (for reference): 0.7539


### Metric choice

Because the classes are moderately imbalanced (~62% class 0, ~38% class 1), accuracy alone could favor the majority class. We therefore use **F1** as the primary metric because it balances precision and recall for the positive/minority class, while **ROC AUC** provides a threshold-independent measure of how well the model ranks the two classes.


## Closing analysis

<!-- Write up: what worked, what didn't, and what you'd try next. Be honest about negative
results - a feature that didn't help, or models that turned out statistically
indistinguishable, reported clearly, is worth more than a tidier-looking story that
isn't quite true. (Replace this cell's text with your own analysis.) -->

**What worked**
- Dropping `boat` / `body` avoided target leakage that would have produced
  unrealistically high scores.
- Treating `Pclass` as categorical (not a continuous scale) matches its meaning.
- A `ColumnTransformer` + `Pipeline` keeps imputation/scaling/encoding inside CV folds.
- Engineered features (`title`, `has_cabin`, `family_size` / `is_alone`) were tested
  via ablation; any lift (or lack of lift) is reported above rather than assumed.

**What did not (or was inconclusive)**
- On a dataset of this size, model-to-model F1 gaps are often smaller than the
  cross-validation standard deviation. When that happens, declaring a single
  “winner” overstates the evidence.

**What we would try next**
- Interaction features (e.g. `Sex × Pclass`) or a small set of binned Age groups.
- We could try looking at the influence of origin country influencing survivability with the missing values being taken care by a Unknown label and the countries along with the unknown label can be one hot encoded.
- Calibrated probabilities if the downstream use needs reliable scores.
- A leaner feature set if ablation showed some engineered columns were pure noise.
